### Summarizing Mani Mama Lecture

* The youtube transcripts were not really good and there were a lot of mistakes. That is why we had to download the video and use openai-whisper library to get it transcribed. Use the transcribe.py to take the MP4 files and output the transcript into a text file.

In [1]:
import chromadb
from langchain_community.document_loaders import YoutubeLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sqlalchemy import text


DB_DIR = "./chromadb"
# Initialize your Chroma vector store
chroma_client = chromadb.PersistentClient(path=DB_DIR)

collection = chroma_client.get_or_create_collection(name="mani_mama_collection")

def load_and_store_youtube_video(transcript_file: str):    
    # 1. Split the transcript into smaller chunks
    # Initialize the splitter
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        separators=["\n\n", "\n", " ", ""]
    )
    # Load the transcript from the file
    with open(transcript_file, "r", encoding="utf-8") as f:
        transcript = f.read()
         # Use split_text to return a list of strings
        chunks = text_splitter.split_text(transcript)

        print(f"Number of chunks: {len(chunks)}")
        split_counter = 0

        # 2. Add documents to the Chroma vector store
        for chunk in chunks:
            collection.add(documents=[chunk], ids=[f"{transcript_file}_{split_counter}"])
            split_counter += 1



C:\Users\VenkyJagannath\AppData\Local\Temp\ipykernel_5404\2142385209.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import YoutubeLoader


In [2]:
# Ingest all the documents for dhyana slokas and chapter 1 
video_urls = [
    "videos/001.txt",
    "videos/002.txt",
    "videos/003.txt",
    "videos/004.txt",
    "videos/005.txt"
]

for url in video_urls:
    load_and_store_youtube_video(url)

Number of chunks: 63
Number of chunks: 53
Number of chunks: 56
Number of chunks: 56
Number of chunks: 71


In [19]:
from langchain_ollama import ChatOllama
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_chroma import Chroma
import textwrap
vector_store = Chroma(collection_name="mani_mama_collection", client=chroma_client)


def query_after_getting_matched_documents(user_query, ollama_model_name="granite4.1:3b"):
    # Create a retriever from the vector store getting top 10 similar documents
    retriever = vector_store.as_retriever(collection_name="mani_mama_collection", search_type="similarity", search_kwargs={"k": 3})

    llm = ChatOllama(model=ollama_model_name, base_url=None)
    # ConversationalRetrievalChain wraps the LLM + retriever
    chain = ConversationalRetrievalChain.from_llm(llm=llm, retriever=retriever, return_source_documents=True)

    result = chain.invoke({"question": user_query, "chat_history":[]})
    print(textwrap.fill(result["answer"], width=70))
    matching_docs = result["source_documents"]
    print("Matching document IDs:")
    for doc in matching_docs:
        print('----------------------')
        print(textwrap.fill(str(doc.id), width=70))
        print('----------------------')

In [ ]:
query = "What is Upasana" 
query_after_getting_matched_documents(query)

Shraddha means unconditional faith or trust in the words of the Guru
(teacher) or scriptures. In the context provided, it refers to
believing wholeheartedly in what the Guru teaches, treating the Guru's
statements as if they are divinely given and not fabrications by the
Guru themselves. It is essential for surrendering oneself completely
to the teachings and guidance of the Guru.
Matching document IDs:
----------------------
videos/005.txt_53
----------------------
----------------------
videos/001.txt_4
----------------------
----------------------
videos/002.txt_15
----------------------


In [ ]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama

ollama_model_name="granite4.1:3b"
llm = ChatOllama(model=ollama_model_name, base_url=None)

template = """
Write a concise summary of the following 

Give the summary in points
{context}
"""
prompt = ChatPromptTemplate.from_template(template)
chain = create_stuff_documents_chain(llm, prompt)
ans = chain.invoke({'context':docs})
print(ans)